In [ ]:
# Install deps
!pip install -q pandas scikit-learn joblib tqdm datasets

In [ ]:
import os, zipfile, requests, sys
url = "https://github.com/vedangvatsa/ai-detection-at-scale/archive/refs/heads/main.zip"
r = requests.get(url)
open("repo.zip", "wb").write(r.content)
with zipfile.ZipFile("repo.zip") as z:
    z.extractall(".")
os.rename("ai-detection-at-scale-main", "ai-detection-at-scale")
os.chdir("ai-detection-at-scale")
sys.path.insert(0, '.')
print("Repo ready")

In [ ]:
# Download data and model assets
!python scripts/download_assets.py

In [ ]:
# Run Beemo benchmark evaluation
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from tool.feature_extractor import extract_features
import joblib

# Load stylometric model
model = joblib.load("models/detector_all.joblib")

# Load dataset from Hugging Face
from datasets import load_dataset
beemo = load_dataset("toloka/beemo")

test_df = beemo['train'].to_pandas()
sample_size = min(2500, len(test_df))
test_sample = test_df.sample(n=sample_size, random_state=42)
print(f"Evaluating on {len(test_sample)} prompts...")

# Extract features and predict
features = []
labels = []
for i, row in test_sample.iterrows():
    h_text = row['human_output']
    m_text = row['model_output']
    
    # Process human text (label = 0)
    if isinstance(h_text, str) and len(h_text.strip()) >= 20:
        feats = extract_features(h_text, extended=False)
        if feats is not None:
            X = np.array([feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                                                'connector_density', 'hedge_density', 'mean_sent_len',
                                                'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']])
            features.append(X)
            labels.append(0)
            
    # Process AI text (label = 1)
    if isinstance(m_text, str) and len(m_text.strip()) >= 20:
        feats = extract_features(m_text, extended=False)
        if feats is not None:
            X = np.array([feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                                                'connector_density', 'hedge_density', 'mean_sent_len',
                                                'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']])
            features.append(X)
            labels.append(1)

features = np.array(features)
labels = np.array(labels)
print(f"Extracted features for {len(labels)} text samples.")

# Predict
probs = model.predict_proba(features)[:, 1]
auc = roc_auc_score(labels, probs)
print(f"Beemo AUC: {auc:.4f}")

# Save results
results = pd.DataFrame([{'benchmark': 'Beemo', 'auc': auc, 'n_samples': len(labels)}])
results.to_csv('/kaggle/working/beemo_results.csv', index=False)
print("Results saved")